# ScamShield BD — Phase 2 Solution (Synthetic Data, Noise Augmentation & Honest Validation)
**Team Name:** Shield

## Declaration
- **Required Libraries & Versions:** `python==3.14.1`, `numpy==2.5.1`, `pandas==3.1.1`, `scipy==1.18.1`, `scikit-learn==1.9.1`, `python-dotenv==1.2.4`
- **External Datasets:** None. Training uses only `sample.csv` plus programmatically generated synthetic/noisy data.
- **Pretrained Models:** None.
- **APIs Used:** None (no API calls; `.env` is empty).
- **Data Generation & Augmentation:** Compositional slot-filling generator (`6,000` rows across English/Bangla/Banglish/Mixed, ~29% multi-turn, hard-negative pairs, real + invented brands) + adversarial noise augmenter (`~30%` of synthetic rows + noisy copies of the `86` real rows) using `SEED = 42`. No rows, labels, or predictions from `test.csv` are ever used in training or augmentation.
- **Methodology & Validation Strategy:**
  - *Preprocessing:* Unicode NFKC normalization; removal of `\u200b`, `\u200c`, `\ufeff` (preserving `\u200d` ZWJ for Bangla); Cyrillic/Greek-to-Latin homoglyph mapping; Latin-only lowercasing; defanged URL replacement (`[URL]`); digit-run replacement (`[NUM]`); `|||` separator preservation.
  - *Features:* `TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), sublinear_tf=True)` + `TfidfVectorizer(analyzer='word', ngram_range=(1, 2), sublinear_tf=True)` combined via `scipy.sparse.hstack`.
  - *Models:* Task A (`LogisticRegression(class_weight='balanced')`), Task B (`OneVsRestClassifier(LogisticRegression(class_weight='balanced'))`, threshold `0.5`), Task C (`LogisticRegression(class_weight='balanced')`) trained on real (`5x` weight) + noisy real (`3x` weight) + synthetic (`1x` weight).
  - *Post-Processing:* `label == 0 -> techniques = 'NONE', action = 0`; non-zero labels with empty technique sets fall back to the single highest-probability technique; sorted `|`-joined codes; confidence = max predicted Task A probability clipped to `[0.05, 0.98]` and rounded to 4 decimals.
  - *Honest Validation:* Evaluated on CLEAN and NOISY sets across (a) real-only 5-fold CV, (b) synthetic-only -> real 86 rows, (c) 5-fold CV on real with synthetic in train folds, and (d) `StratifiedGroupKFold` on synthetic grouped by `family_id`.
- **Random Seeds:** `SEED = 42` for `random`, `numpy`, and `sklearn`.
- **How to Reproduce:** Place `sample.csv`, `test.csv`, and `sample_submission.csv` in the working directory (or `./data/`) and run all cells top to bottom.


In [1]:
import os
import re
import time
import random
import unicodedata
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import f1_score, confusion_matrix
from dotenv import load_dotenv

load_dotenv()

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = 'data' if os.path.isfile(os.path.join('data', 'sample.csv')) else '.'
TECH_CODES = ['T1', 'T2', 'T3', 'T4', 'T5', 'T6']
print('Environment ready. SEED =', SEED, '| DATA_DIR =', DATA_DIR)


Environment ready. SEED = 42 | DATA_DIR = .


## 1. Preprocessing

In [2]:
import re
import unicodedata

# Common Cyrillic / Greek homoglyphs mapped to Latin equivalents
HOMOGLYPH_MAP = {
    # Cyrillic lowercase
    "\u0430": "a",  # а -> a
    "\u0435": "e",  # е -> e
    "\u0451": "e",  # ё -> e
    "\u0456": "i",  # і -> i
    "\u0457": "i",  # ї -> i
    "\u0458": "j",  # ј -> j
    "\u043e": "o",  # о -> o
    "\u0440": "p",  # р -> p
    "\u0441": "c",  # с -> c
    "\u0443": "y",  # у -> y
    "\u0445": "x",  # х -> x
    "\u0455": "s",  # ѕ -> s
    "\u0501": "d",  # ԁ -> d
    "\u051b": "q",  # ԛ -> q
    "\u051d": "w",  # ԝ -> w
    # Cyrillic uppercase
    "\u0410": "A",  # А -> A
    "\u0412": "B",  # В -> B
    "\u0415": "E",  # Е -> E
    "\u041a": "K",  # К -> K
    "\u041c": "M",  # М -> M
    "\u041d": "H",  # Н -> H
    "\u041e": "O",  # О -> O
    "\u0420": "P",  # Р -> P
    "\u0421": "C",  # С -> C
    "\u0422": "T",  # Т -> T
    "\u0425": "X",  # Х -> X
    "\u04ae": "Y",  # Ү -> Y
    "\u0406": "I",  # І -> I
    # Greek lowercase / uppercase
    "\u03b1": "a",  # α -> a
    "\u03b5": "e",  # ε -> e
    "\u03b9": "i",  # ι -> i
    "\u03bf": "o",  # ο -> o
    "\u03c1": "p",  # ρ -> p
    "\u03c2": "c",  # ς -> c
    "\u0391": "A",  # Α -> A
    "\u0392": "B",  # Β -> B
    "\u0395": "E",  # Ε -> E
    "\u0397": "H",  # Η -> H
    "\u0399": "I",  # Ι -> I
    "\u039a": "K",  # Κ -> K
    "\u039c": "M",  # Μ -> M
    "\u039d": "N",  # Ν -> N
    "\u039f": "O",  # Ο -> O
    "\u03a1": "P",  # Ρ -> P
    "\u03a4": "T",  # Τ -> T
    "\u03a7": "X",  # Χ -> X
}
HOMOGLYPH_TRANS = str.maketrans(HOMOGLYPH_MAP)

# Defanged URL pattern: matches hxxp(s)://..., http(s)://..., or domain with [.]
URL_PATTERN = re.compile(
    r"(?:hxxps?|https?|ftp)://\S+|(?:[a-zA-Z0-9-]+\[\.\])+[a-zA-Z]{2,}(?:/\S*)?",
    re.IGNORECASE,
)

# Digit runs (ASCII and Bengali digits)
DIGIT_RUN_PATTERN = re.compile(r"[0-9\u09e6-\u09ef]+(?:[.,][0-9\u09e6-\u09ef]+)*")

# Latin uppercase letters only (preserves Bangla Unicode untouched)
LATIN_UPPER_PATTERN = re.compile(r"[A-Z]+")


def preprocess_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # 1. Unicode NFKC normalization
    t = unicodedata.normalize("NFKC", text)

    # 2. Remove only \u200b, \u200c, \ufeff. Keep \u200d (ZWJ for Bangla).
    t = t.replace("\u200b", "").replace("\u200c", "").replace("\ufeff", "")

    # 3. Map common Cyrillic/Greek homoglyphs to Latin
    t = t.translate(HOMOGLYPH_TRANS)

    # 4. Replace defanged URLs with [URL] token
    t = URL_PATTERN.sub(" [URL] ", t)

    # 5. Lowercase Latin text only (leaves Bangla and [URL] handled cleanly)
    t = LATIN_UPPER_PATTERN.sub(lambda m: m.group(0).lower(), t)
    t = t.replace("[url]", "[URL]")

    # 6. Replace digit runs with [NUM] token
    t = DIGIT_RUN_PATTERN.sub(" [NUM] ", t)

    # 7. Preserve "|||" as a spaced separator token and collapse repeated whitespace
    t = t.replace("|||", " ||| ")
    t = re.sub(r"\s+", " ", t).strip()
    return t


def run_unit_tests():
    # Test 1: Homoglyphs (Cyrillic о, а, і, е, р, с)
    s1 = "Cоngratulаtіоns! Рay Tаkа 500 via bKаsh"
    out1 = preprocess_text(s1)
    assert "congratulations!" in out1, f"Failed homoglyph: {out1}"
    assert "pay taka [NUM] via bkash" in out1, f"Failed homoglyph+num: {out1}"

    # Test 2: Zero-width chars (\u200b, \u200c removed; \u200d kept!)
    s2 = "ব\u200cসে Trx\u200bID\ufeff র\u200d্যাংক"
    out2 = preprocess_text(s2)
    assert "\u200b" not in out2 and "\u200c" not in out2 and "\ufeff" not in out2, f"Failed ZW strip: {out2}"
    assert "\u200d" in out2, f"Failed to preserve ZWJ \\u200d: {out2}"
    assert "trxid" in out2, f"Failed Latin lowercase: {out2}"

    # Test 3: Defanged URLs and conversation separator |||
    s3 = "S: Click hxxps://gp-gift[.]xyz/claim ||| R: Okay 12345"
    out3 = preprocess_text(s3)
    assert "[URL]" in out3, f"Failed URL token: {out3}"
    assert "|||" in out3, f"Failed separator preservation: {out3}"
    assert "[NUM]" in out3, f"Failed NUM token: {out3}"

    # Test 4: Bangla text + Bengali digits
    s4 = "আপনার ৫০০ টাকা ক্যাশ আউট সফল হয়েছে।"
    out4 = preprocess_text(s4)
    assert "আপনার [NUM] টাকা ক্যাশ আউট সফল হয়েছে।" == out4, f"Failed Bangla: {out4}"

    print("All preprocess_text unit tests PASSED!")

run_unit_tests()


All preprocess_text unit tests PASSED!


## 2. Noise Augmenter

In [3]:
import re
import random

HOMOGLYPH_SWAPS = {
    "a": "\u0430",  # Cyrillic а
    "e": "\u0435",  # Cyrillic е
    "i": "\u0456",  # Cyrillic і
    "o": "\u043e",  # Cyrillic о
    "p": "\u0440",  # Cyrillic р
    "c": "\u0441",  # Cyrillic с
    "x": "\u0445",  # Cyrillic х
    "y": "\u0443",  # Cyrillic у
}

LEET_MAP = {
    "a": "4",
    "e": "3",
    "i": "1",
    "o": "0",
    "s": "5",
}

EMOJIS = ["📲", "🔥", "⚠️", "🎁", "💰", "✅", "🚨", "💸", "📦", "🎉"]
NOISE_STRINGS = ["wkrjfejejrn", "mnbvcx", "zzqwpo", "xkcdpl", "asdkjh", "qzxvbn", "lkjsdfoiu"]

BANGLISH_VARIANTS = [
    ("bhul", "vul"),
    ("korun", "koren"),
    ("bolen", "bolun"),
    ("pathan", "pathao"),
    ("taka", "tk"),
    ("apnar", "apnr"),
    ("ekhoni", "ekhn e"),
    ("taratari", "tara tari"),
    ("jitechen", "jitsen"),
    ("হয়েছে", "হয়েছে"),
    ("আয়", "আয়"),
    ("করুন", "করেন"),
    ("যোগ", "জোগ"),
]


def _corrupt_word(word: str, rng: random.Random, prob: float) -> str:
    if len(word) <= 2 or word in {"|||", "S:", "R:", "A:", "B:"} or word.startswith("http") or word.startswith("hxxp"):
        return word

    w = word
    # Bangla/Banglish variant
    if rng.random() < prob * 0.5:
        lw = w.lower()
        for src, dst in BANGLISH_VARIANTS:
            if lw == src:
                w = dst
                break
            elif lw == dst:
                w = src
                break

    # Zero-width char (\u200b or \u200c) inside word (works for both Latin & Bangla)
    if len(w) >= 4 and rng.random() < prob * 0.7:
        idx = rng.randint(1, len(w) - 1)
        zw = rng.choice(["\u200b", "\u200c"])
        w = w[:idx] + zw + w[idx:]

    # Latin-specific corruptions
    if re.search(r"[a-zA-Z]", w):
        chars = list(w)
        r = rng.random()
        if r < prob * 0.35 and len(chars) >= 4:
            # Swap two adjacent chars
            idx = rng.randint(1, len(chars) - 2)
            chars[idx], chars[idx + 1] = chars[idx + 1], chars[idx]
        elif r < prob * 0.6 and len(chars) >= 4:
            # Drop a char
            idx = rng.randint(1, len(chars) - 2)
            chars.pop(idx)
        elif r < prob * 0.85:
            # Repeat a char
            idx = rng.randint(0, len(chars) - 1)
            chars.insert(idx, chars[idx] * rng.randint(1, 2))
        w = "".join(chars)

        # Leet-speak or homoglyph swap
        if rng.random() < prob * 0.45:
            new_chars = []
            for ch in w:
                lch = ch.lower()
                if lch in HOMOGLYPH_SWAPS and rng.random() < 0.3:
                    new_chars.append(HOMOGLYPH_SWAPS[lch])
                elif lch in LEET_MAP and rng.random() < 0.25:
                    new_chars.append(LEET_MAP[lch])
                else:
                    new_chars.append(ch)
            w = "".join(new_chars)

        # Random capitalization
        if rng.random() < prob * 0.4:
            w = "".join(c.upper() if rng.random() < 0.4 else c.lower() for c in w)
    else:
        # Bangla script character drop / repeat / space split
        if len(w) >= 4 and rng.random() < prob * 0.35:
            idx = rng.randint(1, len(w) - 2)
            if rng.random() < 0.5:
                w = w[:idx] + " " + w[idx:]
            else:
                w = w[:idx] + w[idx + 1:]

    return w


def apply_noise(text: str, rng: random.Random, level: float = 0.3) -> str:
    """Apply realistic adversarial noise to English, Banglish, or Bangla text."""
    if not isinstance(text, str) or not text.strip():
        return text

    # Preserve ||| turns
    turns = text.split("|||")
    noisy_turns = []

    for turn in turns:
        tokens = turn.strip().split(" ")
        new_tokens = []
        for tok in tokens:
            # Defanged URL variation
            if "http" in tok.lower() or "[.]" in tok:
                if rng.random() < level:
                    tok = tok.replace("https://", "hxxps://").replace("http://", "hxxp://")
                    if "[.]" not in tok:
                        tok = tok.replace(".", "[.]")
                new_tokens.append(tok)
                continue

            if rng.random() < level:
                tok = _corrupt_word(tok, rng, prob=min(0.9, level + 0.2))
            new_tokens.append(tok)

            # Occasionally drop space between two tokens
            if len(new_tokens) >= 2 and rng.random() < level * 0.12:
                prev = new_tokens.pop()
                new_tokens[-1] = new_tokens[-1] + prev

        # Insert a random noise string
        if len(new_tokens) >= 3 and rng.random() < level * 0.4:
            pos = rng.randint(1, len(new_tokens) - 1)
            new_tokens.insert(pos, rng.choice(NOISE_STRINGS))

        # Insert an emoji
        if rng.random() < level * 0.5:
            pos = rng.choice([0, len(new_tokens)])
            new_tokens.insert(pos, rng.choice(EMOJIS))

        # Extra spaces occasionally
        joiner = "  " if rng.random() < level * 0.15 else " "
        noisy_turns.append(joiner.join(new_tokens))

    return " ||| ".join(noisy_turns)

## 3. Compositional Synthetic Data Generator

In [4]:
import random
import pandas as pd

# Real + Invented Bangladeshi and generic brands/senders
MFS_BRANDS = [
    "bKash", "Nagad", "Rocket", "Upay", "SureCash", "Tap", "Cellfin",
    "PayBondhu", "TakaSwift", "DeshPay", "ChakaPay", "SonarWallet"
]
BANK_BRANDS = [
    "DBBL", "City Bank", "BRAC Bank", "Islami Bank", "Standard Chartered",
    "UCB", "EBL", "Mutual Trust Bank", "NodiBank", "ApexFin BD", "PadmaTrust Bank"
]
COURIER_BRANDS = [
    "Sundarban Courier", "SA Paribahan", "Pathao Courier", "RedX", "Paperfly",
    "Steadfast", "Karatoa", "QuickParcel BD", "SonarCourier", "DhakaExpress"
]
TELCO_BRANDS = [
    "Grameenphone", "GP", "Robi", "Banglalink", "Teletalk", "PadmaTel", "BongoNet"
]
ECOM_BRANDS = [
    "Daraz", "Shwapno", "Aarong", "Chaldal", "Pickaboo", "Rokomari",
    "MeghnaMart", "ShebaHaat", "BongoBazar"
]
EMPLOYER_BRANDS = [
    "Unilever Bangladesh", "Bdjobs", "GrameenTech", "Square Group", "Walton",
    "ApexDigital BD", "EarningHub BD", "DeshWorks"
]
CRYPTO_BRANDS = [
    "Binance", "CoinBangla", "BDTradeX", "AlphaCrypto", "DhakaSignals"
]
PERSON_NAMES = [
    "Rahim", "Karim", "Sadia", "Nusrat", "Tanvir", "Tariq", "Farhan", "Mitu", "Hasan", "Nabila"
]
PHISH_DOMAINS = [
    "verify-bd[.]top", "claim-now[.]xyz", "secure-kyc[.]site", "pay-fee[.]vip",
    "track-parcel[.]top", "bonus-bd[.]info", "vip-earn[.]xyz", "confirm-addr[.]top"
]
LEGIT_DOMAINS = [
    "daraz[.]com[.]bd/track", "payment[.]breb[.]org[.]bd", "mygp[.]grameenphone[.]com",
    "bkash[.]com/help", "sundarbancourier[.]com[.]bd", "bracbank[.]com/portal"
]


def _rand_amt(rng: random.Random) -> str:
    return rng.choice(["45", "50", "100", "250", "300", "500", "750", "1,000", "1,250", "1,500", "2,000", "2,500", "3,500", "5,000", "10,000", "50,000"])


def _rand_phone(rng: random.Random) -> str:
    prefix = rng.choice(["017", "018", "019", "016", "013", "015"])
    return f"{prefix}XXXXXX{rng.randint(10, 99)}"


def _rand_trx(rng: random.Random) -> str:
    chars = "ABCDEFGHJKLMNPQRSTUVWXYZ23456789"
    return "".join(rng.choice(chars) for _ in range(8))


def _rand_otp(rng: random.Random) -> str:
    return str(rng.randint(100000, 999999))


def _rand_parcel(rng: random.Random) -> str:
    return f"{rng.choice(['SP', 'SC', 'DZ', 'PF', 'CN', 'RX'])}{rng.randint(1000, 99999)}"


def _build_slots(rng: random.Random) -> dict:
    return {
        "mfs": rng.choice(MFS_BRANDS),
        "bank": rng.choice(BANK_BRANDS),
        "courier": rng.choice(COURIER_BRANDS),
        "telco": rng.choice(TELCO_BRANDS),
        "ecom": rng.choice(ECOM_BRANDS),
        "emp": rng.choice(EMPLOYER_BRANDS),
        "crypto": rng.choice(CRYPTO_BRANDS),
        "name": rng.choice(PERSON_NAMES),
        "amt": _rand_amt(rng),
        "bal": str(rng.randint(500, 25000)),
        "phone": _rand_phone(rng),
        "trx": _rand_trx(rng),
        "otp": _rand_otp(rng),
        "parcel": _rand_parcel(rng),
        "purl": f"hxxps://{rng.choice(PHISH_DOMAINS)}/{rng.choice(['verify', 'claim', 'pay', 'update'])}",
        "lurl": f"https://{rng.choice(LEGIT_DOMAINS)}",
    }


def _gen_single_templates(s: dict) -> list:
    """Return list of (family_id, language, label, techniques, action, is_hard_negative, text)."""
    rows = []

    # ==================== CLASS 0: LEGITIMATE (HARD LEGIT + PERSONAL) ====================
    # 1. Legit OTP with do-not-share warning (4 languages)
    rows.append(("c0_otp_warn", "english", 0, "NONE", 0, True,
                 f"{s['mfs']} OTP is {s['otp']}. Do not share this code with anyone, including {s['mfs']} staff. Valid 3 min."))
    rows.append(("c0_otp_warn", "bangla", 0, "NONE", 0, True,
                 f"আপনার {s['mfs']} ওটিপি {s['otp']}। এই কোড কারো সাথে শেয়ার করবেন না, এমনকি {s['mfs']} কর্মীকেও না।"))
    rows.append(("c0_otp_warn", "banglish", 0, "NONE", 0, True,
                 f"{s['bank']} OTP: {s['otp']}. Eta kauke bolben na. Bank er lokeo PIN ba OTP chay na. Meyad 3 minute."))
    rows.append(("c0_otp_warn", "mixed", 0, "NONE", 0, True,
                 f"{s['mfs']} login code {s['otp']} for Tk {s['amt']} payment. Kauke share korben na."))

    # 2. Legit Cash-out / Receive / Debit confirmation
    rows.append(("c0_tx_confirm", "english", 0, "NONE", 0, True,
                 f"{s['mfs']}: You have received Tk {s['amt']} from {s['phone']}. Fee Tk 0.00. Balance Tk {s['bal']}. TrxID {s['trx']}."))
    rows.append(("c0_tx_confirm", "bangla", 0, "NONE", 0, True,
                 f"আপনার {s['amt']} টাকা ক্যাশ আউট সফল হয়েছে। বর্তমান ব্যালেন্স {s['bal']} টাকা। TrxID {s['trx']}।"))
    rows.append(("c0_tx_confirm", "banglish", 0, "NONE", 0, True,
                 f"Dhonyobad! Apnar Tk {s['amt']} recharge successful. Balance Tk {s['bal']}. TrxID {s['trx']}."))
    rows.append(("c0_tx_confirm", "mixed", 0, "NONE", 0, True,
                 f"{s['bank']}: BDT {s['amt']} debited from A/C XXXX{s['otp'][:4]} at {s['ecom']}. Avl bal BDT {s['bal']}."))

    # 3. Legit Delivery / Parcel updates (no fee demanded)
    rows.append(("c0_parcel_legit", "english", 0, "NONE", 0, True,
                 f"Your {s['ecom']} order #{s['parcel']} is out for delivery today. Track: {s['lurl']}"))
    rows.append(("c0_parcel_legit", "bangla", 0, "NONE", 0, True,
                 f"{s['courier']}: আপনার পার্সেল {s['parcel']} আজ ডেলিভারি হবে। শাখা থেকে সংগ্রহ করতে এই এসএমএস ও এনআইডি আনুন।"))
    rows.append(("c0_parcel_legit", "banglish", 0, "NONE", 0, True,
                 f"{s['courier']}: Parcel {s['parcel']} Chattogram hub e pouchheche. Branch theke collect korun."))
    rows.append(("c0_parcel_legit", "mixed", 0, "NONE", 0, True,
                 f"{s['ecom']} order #{s['parcel']} delivered. Cash on delivery Tk {s['amt']} paid. Thank you."))

    # 4. Legit Security Alert / Scam Warning / Bill Reminder / Real Cashback Confirmation
    rows.append(("c0_sec_alert", "english", 0, "NONE", 0, True,
                 f"Security alert: New device login on {s['mfs']}. If this was you, ignore. We never ask for your PIN or OTP."))
    rows.append(("c0_sec_alert", "banglish", 0, "NONE", 0, True,
                 f"Sabdhan! Keu jodi {s['mfs']} agent bole PIN ba OTP chay, kokhono diben na. Eta scam."))
    rows.append(("c0_bill_remind", "mixed", 0, "NONE", 0, True,
                 f"Reminder: Your {s['telco']} bill of Tk {s['amt']} is due on 3 Dec. Pay via official app or {s['lurl']}"))
    rows.append(("c0_cashback_real", "bangla", 0, "NONE", 0, True,
                 f"আপনার {s['ecom']} পেমেন্টের জন্য Tk {s['amt']} ক্যাশব্যাক আপনার {s['mfs']} অ্যাকাউন্টে যোগ হয়েছে।"))

    # 5. Ordinary Personal Chat / Appointment
    rows.append(("c0_personal", "english", 0, "NONE", 0, False,
                 f"Hi {s['name']}, meeting moved to 3 PM. Please join the calendar invite link."))
    rows.append(("c0_personal", "bangla", 0, "NONE", 0, False,
                 f"{s['name']}, কাল সকাল ১০টায় ক্লাস হবে। স্যার ইমেইল করেছেন।"))
    rows.append(("c0_personal", "banglish", 0, "NONE", 0, False,
                 f"Mama, ajke raate bari asbo. Bazar theke kichu lagbe? {s['name']} ke bolo."))
    rows.append(("c0_personal", "mixed", 0, "NONE", 0, False,
                 f"Ami ekhon {s['mfs']} e {s['amt']} taka pathaisi, dekh to ashche kina."))

    # ==================== CLASS 1: MFS / OTP SCAM ====================
    # 1. Account lock / KYC threat asking for PIN/OTP/NID
    rows.append(("c1_kyc_lock", "english", 1, "T1|T2|T5", 1, True,
                 f"{s['mfs']} Helpline: Unusual login detected. Reply with your 6-digit OTP and PIN within 1 hour or account will be blocked."))
    rows.append(("c1_kyc_lock", "bangla", 1, "T1|T2|T5", 1, True,
                 f"প্রিয় গ্রাহক, আপনার {s['mfs']} অ্যাকাউন্ট ২৪ ঘণ্টার মধ্যে বন্ধ হয়ে যাবে। চালু রাখতে পিন ও ওটিপি নম্বর পাঠান।"))
    rows.append(("c1_kyc_lock", "banglish", 1, "T1|T2|T5", 1, True,
                 f"Dear {s['mfs']} user, KYC update na korle account band hobe. Update korte NID number o PIN pathan."))
    rows.append(("c1_kyc_lock", "mixed", 1, "T1|T2|T5", 1, True,
                 f"{s['bank']} Security: Card ending {s['otp'][:4]} flagged. Share the OTP you just received. Last warning."))

    # 2. Mistaken transfer claim asking for money back / PIN
    rows.append(("c1_mistake_tx", "banglish", 1, "T1|T2|T5", 1, True,
                 f"Ami {s['mfs']} agent, bhul number-e {s['amt']} taka pathaisi. Ferot din, PIN ta bolen."))
    rows.append(("c1_mistake_tx", "mixed", 1, "T3|T5", 2, True,
                 f"Bhai ekta bhul hoye geche. Amar {s['amt']} taka vul kore apnar {s['mfs']} e chole gese. Please ferot den {s['phone']} e. Ami bipode."))
    rows.append(("c1_mistake_tx", "bangla", 1, "T1|T2|T5", 1, True,
                 f"আমি {s['mfs']} এজেন্ট, ভুল করে আপনার নম্বরে {s['amt']} টাকা গেছে। টাকা ফেরত দিতে ওটিপি কোডটি বলুন।"))
    rows.append(("c1_mistake_tx", "english", 1, "T1|T3|T5", 2, True,
                 f"Mistaken transfer of Tk {s['amt']} sent to your {s['mfs']}. Return immediately to {s['phone']} or police action will be taken."))

    # 3. Card / Refund team phishing
    rows.append(("c1_card_refund", "english", 1, "T1|T2", 1, False,
                 f"{s['ecom']} refund team: send card number, expiry and CVV to receive your Tk {s['amt']} refund."))
    rows.append(("c1_card_refund", "banglish", 1, "T1|T2", 1, False,
                 f"{s['mfs']} bonus: apni Tk {s['amt']} peyechen. Wallet e add korte PIN o card CVV din."))

    # ==================== CLASS 2: PRIZE / LOTTERY SCAM ====================
    rows.append(("c2_lottery_link", "english", 2, "T4|T5", 3, False,
                 f"Congratulations! You won Tk {s['amt']} cashback. Click {s['purl']} to claim within 1 hour."))
    rows.append(("c2_lottery_link", "bangla", 2, "T4|T5", 3, False,
                 f"অভিনন্দন! আপনি {s['amt']} টাকা লটারি জিতেছেন। পুরস্কার পেতে এখনই লিংকে ক্লিক করুন: {s['purl']}"))
    rows.append(("c2_lottery_link", "banglish", 2, "T4|T5", 3, False,
                 f"Apni {s['amt']} taka jitechen! Claim korte link e click korun: {s['purl']}"))
    rows.append(("c2_lottery_link", "mixed", 2, "T1|T4|T5", 3, True,
                 f"Free 10GB data gift from {s['telco']} for you! Tap {s['purl']} now, offer expires in 1 hour."))

    rows.append(("c2_prize_fee", "english", 2, "T1|T3|T4", 2, False,
                 f"You won Tk {s['amt']} from {s['telco']} lucky draw! Pay Tk 300 processing fee via {s['mfs']} to claim."))
    rows.append(("c2_prize_fee", "banglish", 2, "T1|T4", 5, False,
                 f"Congratulations! Apnar number ti {s['telco']} Eid lucky draw te select hoyeche. iPhone jitar jonno {s['phone']} e WhatsApp korun."))
    rows.append(("c2_prize_app", "mixed", 2, "T1|T4", 4, False,
                 f"{s['mfs']} anniversary: spin and win Tk {s['amt']}! Download the spin app: {s['purl']}/app.apk"))
    rows.append(("c2_prize_details", "english", 2, "T4", 1, False,
                 f"Lucky draw winner! Reply with your full name, NID and address to receive a brand new car."))

    # ==================== CLASS 3: FAKE JOB / INVESTMENT ====================
    rows.append(("c3_task_earn", "english", 3, "T6", 4, False,
                 f"Part-time job: like YouTube videos, earn Tk {s['amt']} per task. Join Telegram: {s['purl']}"))
    rows.append(("c3_task_earn", "bangla", 3, "T6", 4, False,
                 f"ঘরে বসে প্রতিদিন {s['amt']} টাকা আয় করুন। সহজ লাইক ও কমেন্টের কাজ। আজই হোয়াটসঅ্যাপে যোগ দিন।"))
    rows.append(("c3_task_earn", "banglish", 3, "T6", 4, False,
                 f"Ghor bose din e {s['amt']} taka income. Facebook page like korle taka. WhatsApp e join korun."))
    rows.append(("c3_task_earn", "mixed", 3, "T6", 4, False,
                 f"Task: buy products on {s['ecom']} via our link, we refund + 20% commission. Complete 5 tasks, get Tk {s['amt']}."))

    rows.append(("c3_invest_fee", "english", 3, "T3|T6", 2, False,
                 f"Double your money in 7 days! Invest Tk {s['amt']}, get double guaranteed. Send via {s['mfs']} {s['phone']}."))
    rows.append(("c3_invest_fee", "mixed", 3, "T1|T3|T6", 2, False,
                 f"Congratulations, selected for Data Entry Officer at {s['emp']}. Pay Tk {s['amt']} registration fee to confirm."))
    rows.append(("c3_crypto", "english", 3, "T5|T6", 4, False,
                 f"{s['crypto']} VIP signals group - 300% weekly profit guaranteed! Join now: {s['purl']}"))
    rows.append(("c3_abroad_job", "banglish", 3, "T3|T6", 5, False,
                 f"Abroad job in Malaysia, visa fee Tk {s['amt']}. 100% guaranteed. Call {s['phone']} ekhoni."))

    # ==================== CLASS 4: DELIVERY / PARCEL PHISHING ====================
    rows.append(("c4_parcel_fee", "english", 4, "T1|T3", 2, True,
                 f"{s['courier']}: Parcel {s['parcel']} held at hub. Pay Tk {s['amt']} storage fee: {s['purl']}"))
    rows.append(("c4_parcel_fee", "bangla", 4, "T3", 2, True,
                 f"আপনার পার্সেল {s['parcel']} আটকে আছে। ডেলিভারি চার্জ {s['amt']} টাকা পরিশোধ করতে লিংকে ক্লিক করুন: {s['purl']}"))
    rows.append(("c4_parcel_fee", "banglish", 4, "T3", 2, True,
                 f"Parcel {s['parcel']} atke ase. Delivery fee Tk {s['amt']} dite link e jan: {s['purl']}"))
    rows.append(("c4_customs", "mixed", 4, "T1|T3", 2, True,
                 f"Customs hold: your parcel from China needs Tk {s['amt']} duty. Pay via {s['mfs']} {s['phone']} to release."))

    rows.append(("c4_addr_link", "english", 4, "T1|T5", 3, True,
                 f"Courier notice: package #{s['parcel']} requires address confirmation. Click {s['purl']} within 24h."))
    rows.append(("c4_addr_link", "bangla", 4, "T1|T5", 3, True,
                 f"{s['courier']}: ডেলিভারি ব্যর্থ হয়েছে। ঠিকানা আপডেট করতে ১২ ঘণ্টার মধ্যে {s['purl']} এ যান।"))
    rows.append(("c4_addr_call", "banglish", 4, "T1|T5", 5, True,
                 f"{s['courier']}: Apnar parcel er address vul. Call korun {s['phone']} ekhoni."))

    # ==================== CLASS 5: PROMOTIONAL SPAM ====================
    rows.append(("c5_promo_offer", "english", 5, "NONE", 0, True,
                 f"Get 10% cashback on your next top-up with code EID10. Valid today only!"))
    rows.append(("c5_promo_offer", "bangla", 5, "NONE", 0, True,
                 f"ঈদ উপলক্ষে {s['ecom']}-এ সব পণ্যে ৩০% ছাড়! আজই কিনুন, শর্ত প্রযোজ্য।"))
    rows.append(("c5_promo_offer", "banglish", 5, "NONE", 0, True,
                 f"Eid offer! {s['ecom']} e sob item e 30% discount. Shop now: {s['lurl']}"))
    rows.append(("c5_promo_offer", "mixed", 5, "NONE", 0, True,
                 f"{s['telco']} Mega Offer! Buy 1GB get 1GB free for Tk {s['amt']}. Dial *121*3#"))
    rows.append(("c5_promo_app", "english", 5, "NONE", 0, False,
                 f"Install {s['ecom']} app and get Tk 50 off your first order. Use code WELCOME50."))
    rows.append(("c5_promo_biz", "mixed", 5, "NONE", 0, False,
                 f"Book your dream apartment in Bashundhara, easy installment. Call {s['phone']}"))
    rows.append(("c5_promo_course", "english", 5, "NONE", 0, False,
                 f"Learn English in 30 days! Admission going on at {s['emp']}. Call now."))

    return rows


def _gen_multiturn_templates(rng: random.Random, s: dict) -> list:
    """Generate multi-turn conversations (2-12 turns) with delayed intent, small talk, or strategy shifts."""
    rows = []
    spk_s, spk_r = rng.choice([("S", "R"), ("Caller", "You"), ("Agent", "Customer"), ("A", "B")])

    # Class 0 multi-turn: personal chat or legit support
    t_c0_1 = [
        f"{spk_s}: Bhai, kemon asos?",
        f"{spk_r}: Alhamdulillah bhalo. Tui?",
        f"{spk_s}: Bhalo. {s['amt']} taka lagbe, Sunday ferot dibo?",
        f"{spk_r}: Ha thik ase, {s['mfs']} e pathacchi.",
        f"{spk_s}: Ok thanks!"
    ]
    rows.append(("c0_mt_chat", "banglish", 0, "NONE", 0, True, len(t_c0_1), " ||| ".join(t_c0_1)))

    t_c0_2 = [
        f"{spk_r}: I forgot my {s['mfs']} PIN. How can I reset it?",
        f"{spk_s}: Dial *247# and choose Reset PIN.",
        f"{spk_s}: Never share your PIN or OTP with anyone, including our support staff.",
        f"{spk_r}: Thank you!"
    ]
    rows.append(("c0_mt_support", "english", 0, "NONE", 0, True, len(t_c0_2), " ||| ".join(t_c0_2)))

    t_c0_3 = [
        f"{spk_r}: আমার পার্সেল {s['parcel']} কি আজ আসবে?",
        f"{spk_s}: হ্যাঁ, রাইডার বিকেল ৫টার দিকে ডেলিভারি করবে।",
        f"{spk_r}: ঠিক আছে, আমি ক্যাশ অন ডেলিভারি দেবো।"
    ]
    rows.append(("c0_mt_delivery", "bangla", 0, "NONE", 0, True, len(t_c0_3), " ||| ".join(t_c0_3)))

    # Class 1 multi-turn: trust building -> mistaken transfer -> PIN/OTP ask (up to 7 turns)
    t_c1_1 = [
        f"{spk_s}: Assalamu alaikum, ami {s['mfs']} office theke bolchi.",
        f"{spk_r}: Wa alaikumussalam, bolun.",
        f"{spk_s}: Apnar account e technical check cholche. Bhul kore {s['amt']} taka chole gese.",
        f"{spk_r}: Ami to kono SMS pai nai.",
        f"{spk_s}: Wait korun, ekta 6 digit code asbe.",
        f"{spk_s}: Account verify korte oi OTP ar apnar PIN ta bolen, na hole account block hobe."
    ]
    rows.append(("c1_mt_agent", "banglish", 1, "T1|T2|T5", 1, True, len(t_c1_1), " ||| ".join(t_c1_1)))

    t_c1_2 = [
        f"{spk_s}: Hello, I am calling from {s['bank']} security.",
        f"{spk_r}: Yes, is there an issue?",
        f"{spk_s}: Someone tried to withdraw Tk {s['amt']} from your card.",
        f"{spk_r}: Oh no! What should I do?",
        f"{spk_s}: To cancel it, quickly tell me the OTP you just received and your card CVV."
    ]
    rows.append(("c1_mt_bank", "english", 1, "T1|T2|T5", 1, True, len(t_c1_2), " ||| ".join(t_c1_2)))

    # Class 2 multi-turn: prize bait -> fee or link
    t_c2_1 = [
        f"{spk_s}: Hello {s['name']}, you have been selected for our {s['ecom']} loyalty reward!",
        f"{spk_r}: Really? What reward?",
        f"{spk_s}: A free smartphone worth 45000 BDT.",
        f"{spk_r}: How do I get it?",
        f"{spk_s}: Just pay Tk {s['amt']} delivery charge via {s['mfs']} to {s['phone']}."
    ]
    rows.append(("c2_mt_prize", "english", 2, "T1|T3|T4", 2, False, len(t_c2_1), " ||| ".join(t_c2_1)))

    t_c2_2 = [
        f"{spk_s}: অভিনন্দন! আপনি {s['telco']} লটারিতে {s['amt']} টাকা জিতেছেন।",
        f"{spk_r}: সত্যি নাকি? কীভাবে পাবো?",
        f"{spk_s}: এই লিংকে গিয়ে ফর্ম পূরণ করুন: {s['purl']}। ১ ঘণ্টার মধ্যে ক্লেইম করুন।"
    ]
    rows.append(("c2_mt_lottery_bn", "bangla", 2, "T1|T4|T5", 3, False, len(t_c2_2), " ||| ".join(t_c2_2)))

    t_c4_2 = [
        f"{spk_s}: আপনার {s['courier']} পার্সেল #{s['parcel']} কাস্টমসে আটকে আছে।",
        f"{spk_r}: আমি তো কিছু অর্ডার করিনি।",
        f"{spk_s}: আপনার নামে এসেছে। ছাড়াতে {s['amt']} টাকা ফি দিন: {s['purl']}"
    ]
    rows.append(("c4_mt_customs_bn", "bangla", 4, "T1|T3|T5", 2, True, len(t_c4_2), " ||| ".join(t_c4_2)))

    t_c5_2 = [
        f"{spk_s}: {s['ecom']} মেগা সেলে আজ সব পণ্যে ৪০% ছাড়!",
        f"{spk_s}: অফার নিতে ভিজিট করুন {s['lurl']}"
    ]
    rows.append(("c5_mt_promo_bn", "bangla", 5, "NONE", 0, True, len(t_c5_2), " ||| ".join(t_c5_2)))

    t_c1_3 = [
        f"{spk_s}: হ্যালো, আমি {s['mfs']} কাস্টমার কেয়ার থেকে বলছি।",
        f"{spk_r}: জ্বী বলুন।",
        f"{spk_s}: আপনার একাউন্টে সমস্যা হয়েছে, দ্রুত আপনার পিন ও ওটিপি দিন।"
    ]
    rows.append(("c1_mt_agent_bn", "bangla", 1, "T1|T2|T5", 1, True, len(t_c1_3), " ||| ".join(t_c1_3)))

    t_c0_4 = [
        f"{spk_r}: Bhai, {s['ecom']} order #{s['parcel']} ki ashche?",
        f"{spk_s}: Ha, rider call korse, 5 tay delivery hobe.",
        f"{spk_r}: Ok, cash on delivery dibo."
    ]
    rows.append(("c0_mt_delivery_mix", "mixed", 0, "NONE", 0, True, len(t_c0_4), " ||| ".join(t_c0_4)))

    t_c4_3 = [
        f"{spk_s}: {s['courier']}: Parcel #{s['parcel']} address incomplete.",
        f"{spk_r}: Ki korte hobe?",
        f"{spk_s}: Confirm address within 12h at {s['purl']} or parcel return hobe."
    ]
    rows.append(("c4_mt_addr_mix", "mixed", 4, "T1|T5", 3, True, len(t_c4_3), " ||| ".join(t_c4_3)))

    # Class 3 multi-turn: job interview -> strategy shift to registration fee / Telegram task
    t_c3_1 = [
        f"{spk_s}: Hi, I saw your CV on {s['emp']}. We have a work-from-home position.",
        f"{spk_r}: What is the job and salary?",
        f"{spk_s}: Data entry, 35,000 BDT per month.",
        f"{spk_r}: Sounds good, how do I start?",
        f"{spk_s}: You need to deposit Tk {s['amt']} as security/registration fee first."
    ]
    rows.append(("c3_mt_jobfee", "english", 3, "T1|T3|T6", 2, False, len(t_c3_1), " ||| ".join(t_c3_1)))

    t_c3_2 = [
        f"{spk_s}: Hello, part-time online kaj korte chan?",
        f"{spk_r}: Ki kaj?",
        f"{spk_s}: YouTube video like korle daily {s['amt']} taka income.",
        f"{spk_s}: First e Telegram group e join korun: {s['purl']}"
    ]
    rows.append(("c3_mt_task", "mixed", 3, "T6", 4, False, len(t_c3_2), " ||| ".join(t_c3_2)))

    # Class 4 multi-turn: parcel notice -> customs fee or address link
    t_c4_1 = [
        f"{spk_s}: Hello, this is {s['courier']}. You have a package #{s['parcel']} from abroad.",
        f"{spk_r}: I didn't order anything.",
        f"{spk_s}: It is registered under your name at customs.",
        f"{spk_r}: What do I need to do?",
        f"{spk_s}: Pay Tk {s['amt']} customs duty today or it will be returned."
    ]
    rows.append(("c4_mt_customs", "english", 4, "T1|T3|T5", 2, True, len(t_c4_1), " ||| ".join(t_c4_1)))

    # Class 5 multi-turn: promo dialogue
    t_c5_1 = [
        f"{spk_s}: Sir, hair fall problem? Our herbal oil is very popular.",
        f"{spk_s}: 40% discount today only, order now at {s['lurl']}."
    ]
    rows.append(("c5_mt_promo", "english", 5, "NONE", 0, True, len(t_c5_1), " ||| ".join(t_c5_1)))

    # Longer 8-10 turn conversation (Class 1 or Class 3) to cover up to 12 turns
    t_long = [
        f"{spk_s}: Assalamu alaikum, apnar sathe kotha bola jabe?",
        f"{spk_r}: Ji bolun, ke bolchen?",
        f"{spk_s}: Ami {s['emp']} HR theke {s['name']}.",
        f"{spk_r}: Oh accha, ki bishoye?",
        f"{spk_s}: Apni amader online task program e select hoyechen.",
        f"{spk_r}: Eta ki rokom kaj?",
        f"{spk_s}: Ghor bose daily {s['amt']} taka income kora jabe.",
        f"{spk_r}: Kono fee lagbe?",
        f"{spk_s}: Ji, shudhu {s['amt']} taka registration fee diye WhatsApp group e join korte hobe."
    ]
    rows.append(("c3_mt_long", "banglish", 3, "T1|T3|T6", 4, False, len(t_long), " ||| ".join(t_long)))

    return rows


def generate_synthetic_dataset(target_rows: int = 6000, seed: int = 42) -> pd.DataFrame:
    rng = random.Random(seed)
    # Target counts per class (extra weight for 0 and 5 as requested)
    target_per_class = {
        0: int(target_rows * 0.20),  # 1200
        1: int(target_rows * 0.155), # 930
        2: int(target_rows * 0.155), # 930
        3: int(target_rows * 0.155), # 930
        4: int(target_rows * 0.155), # 930
        5: int(target_rows * 0.18),  # 1080
    }

    buckets = {k: [] for k in range(6)}

    # Keep generating batches until all class buckets are full
    while any(len(buckets[k]) < target_per_class[k] for k in range(6)):
        slots = _build_slots(rng)
        single_rows = _gen_single_templates(slots)
        mt_rows = _gen_multiturn_templates(rng, slots)

        # ~65% single, ~35% multi-turn weighting, boosting non-English slightly for ~25% language balance
        for fam, lang, lbl, tech, act, hard, txt in single_rows:
            lang_p = 0.40 if lang == "english" else 0.72
            if len(buckets[lbl]) < target_per_class[lbl] and rng.random() < lang_p:
                buckets[lbl].append({
                    "text": txt,
                    "label": lbl,
                    "techniques": "|".join(sorted(tech.split("|"))) if tech != "NONE" else "NONE",
                    "action": act,
                    "family_id": fam,
                    "language": lang,
                    "n_turns": 1,
                    "is_hard_negative": hard,
                })

        for fam, lang, lbl, tech, act, hard, n_t, txt in mt_rows:
            if len(buckets[lbl]) < target_per_class[lbl] and rng.random() < 0.85:
                buckets[lbl].append({
                    "text": txt,
                    "label": lbl,
                    "techniques": "|".join(sorted(tech.split("|"))) if tech != "NONE" else "NONE",
                    "action": act,
                    "family_id": fam,
                    "language": lang,
                    "n_turns": n_t,
                    "is_hard_negative": hard,
                })

    all_rows = []
    for k in range(6):
        all_rows.extend(buckets[k][:target_per_class[k]])

    rng.shuffle(all_rows)
    df = pd.DataFrame(all_rows)
    df.insert(0, "id", range(100001, 100001 + len(df)))
    return df

## 4. Honest Validation, Phase 2 Training, Prediction & Validation

In [5]:
import os
import sys
import pandas as pd

REQUIRED_COLS = ["id", "label", "techniques", "action", "confidence"]
VALID_CODES = {"T1", "T2", "T3", "T4", "T5", "T6"}


def get_data_dir():
    return "data" if os.path.isfile(os.path.join("data", "sample.csv")) else "."


def check_techniques(val: str) -> bool:
    if not isinstance(val, str):
        return False
    if val != val.strip() or not val or " " in val:
        return False
    if val == "NONE":
        return True
    parts = val.split("|")
    if "NONE" in parts:
        return False
    if any(p not in VALID_CODES for p in parts):
        return False
    if len(parts) != len(set(parts)):
        return False
    if parts != sorted(parts):
        return False
    return True


def validate_submission(sub_path: str = "submission.csv") -> bool:
    data_dir = DATA_DIR
    ref_path = os.path.join(data_dir, "sample_submission.csv")
    test_path = os.path.join(data_dir, "test.csv")

    failures = []

    if not os.path.isfile(sub_path):
        print(f"FAIL: File not found: {sub_path}")
        return False

    sub = pd.read_csv(sub_path, keep_default_na=False)
    ref = pd.read_csv(ref_path, usecols=["id"])
    test = pd.read_csv(test_path, usecols=["id"])

    # 1. Header matches exactly
    if list(sub.columns) != REQUIRED_COLS:
        failures.append(f"Header mismatch: got {list(sub.columns)}, expected {REQUIRED_COLS}")

    # 2. Row count & ID set / order / duplicates
    if len(sub) != len(test):
        failures.append(f"Row count mismatch: got {len(sub)}, expected {len(test)}")
    if len(sub) != len(ref):
        failures.append(f"Row count mismatch vs sample_submission: got {len(sub)}, expected {len(ref)}")

    try:
        sub_ids = pd.to_numeric(sub["id"], errors="raise")
        if sub_ids.duplicated().any():
            failures.append(f"Duplicate IDs found: {sub_ids.duplicated().sum()}")
        if len(sub_ids) == len(test) and not (sub_ids.values == test["id"].values).all():
            failures.append("IDs do not match test.csv order or values")
    except Exception as e:
        failures.append(f"ID parsing error: {e}")

    # 3. No nulls or blank strings anywhere
    for col in REQUIRED_COLS:
        if col in sub.columns:
            blank_cnt = (sub[col].astype(str).str.strip() == "").sum()
            if blank_cnt > 0:
                failures.append(f"Column '{col}' has {blank_cnt} blank/null values")

    # 4. label and action are integers in 0-5
    for col in ["label", "action"]:
        if col in sub.columns:
            try:
                vals = pd.to_numeric(sub[col], errors="raise")
                if (vals % 1 != 0).any():
                    failures.append(f"Column '{col}' has non-integer values")
                if (~vals.isin([0, 1, 2, 3, 4, 5])).any():
                    failures.append(f"Column '{col}' has values outside 0-5")
            except Exception as e:
                failures.append(f"Column '{col}' parsing error: {e}")

    # 5. techniques validation
    if "techniques" in sub.columns:
        bad_tech = (~sub["techniques"].apply(check_techniques)).sum()
        if bad_tech > 0:
            failures.append(f"Column 'techniques' has {bad_tech} invalid entries")

    # 6. confidence numeric in [0, 1] and not NaN
    if "confidence" in sub.columns:
        try:
            confs = pd.to_numeric(sub["confidence"], errors="raise")
            if confs.isna().any() or ((confs < 0.0) | (confs > 1.0)).any():
                failures.append("Column 'confidence' has NaN or values outside [0, 1]")
        except Exception as e:
            failures.append(f"Column 'confidence' parsing error: {e}")

    if failures:
        print("FAIL:")
        for f in failures:
            print("  -", f)
        return False
    else:
        print("PASS")
        return True

import os
import random
import time
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import f1_score, confusion_matrix


SEED = 42
TECH_CODES = ["T1", "T2", "T3", "T4", "T5", "T6"]


def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)


def encode_techniques(series: pd.Series) -> np.ndarray:
    y = np.zeros((len(series), len(TECH_CODES)), dtype=int)
    for i, val in enumerate(series.astype(str)):
        parts = val.split("|")
        for j, code in enumerate(TECH_CODES):
            if code in parts:
                y[i, j] = 1
    return y


def postprocess_predictions(labels, tech_probs, actions, confs):
    out_techs = []
    out_actions = actions.copy()
    for i in range(len(labels)):
        if labels[i] == 0:
            out_techs.append("NONE")
            out_actions[i] = 0
        else:
            active = [TECH_CODES[j] for j in range(len(TECH_CODES)) if tech_probs[i, j] >= 0.5]
            if not active:
                active = [TECH_CODES[int(np.argmax(tech_probs[i]))]]
            out_techs.append("|".join(sorted(active)))
    out_confs = np.round(np.clip(confs, 0.05, 0.98), 4)
    return labels, out_techs, out_actions, out_confs


def fit_vectorizers(texts):
    char_vec = TfidfVectorizer(
        analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, dtype=np.float32
    )
    word_vec = TfidfVectorizer(
        analyzer="word", ngram_range=(1, 2), sublinear_tf=True, dtype=np.float32
    )
    X_char = char_vec.fit_transform(texts)
    X_word = word_vec.fit_transform(texts)
    return char_vec, word_vec, hstack([X_char, X_word]).tocsr()


def transform_vectorizers(char_vec, word_vec, texts):
    return hstack([char_vec.transform(texts), word_vec.transform(texts)]).tocsr()


def fit_and_predict(tr_texts, tr_lbl, tr_tech, tr_act, va_texts, sample_weight=None):
    cv_c, cv_w, X_tr = fit_vectorizers(tr_texts)
    X_va = transform_vectorizers(cv_c, cv_w, va_texts)

    ca = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)
    ca.fit(X_tr, tr_lbl, sample_weight=sample_weight)
    pa = ca.predict_proba(X_va)
    pred_lbl = np.argmax(pa, axis=1)
    pred_conf = np.max(pa, axis=1)

    cb = OneVsRestClassifier(
        LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)
    )
    cb.fit(X_tr, tr_tech)
    prob_tech = cb.predict_proba(X_va)

    cc = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)
    cc.fit(X_tr, tr_act, sample_weight=sample_weight)
    pred_act = cc.predict(X_va)

    return postprocess_predictions(pred_lbl, prob_tech, pred_act, pred_conf)


def compute_metrics(y_lbl, y_tech, y_act, p_lbl, p_tech_strs, p_act):
    p_tech_bin = encode_techniques(pd.Series(p_tech_strs))
    f1_a = f1_score(y_lbl, p_lbl, average="macro")
    f1_b = float(np.mean([f1_score(y_tech[:, j], p_tech_bin[:, j], zero_division=0) for j in range(6)]))
    f1_c = f1_score(y_act, p_act, average="macro")
    return f1_a, f1_b, f1_c


def run_phase2():
    set_seed(SEED)
    rng = random.Random(SEED)
    t0 = time.time()
    data_dir = get_data_dir()

    # 1. Load real sample.csv & build noisy copies
    real_df = pd.read_csv(os.path.join(data_dir, "sample.csv"))
    real_clean_texts = real_df["text"].fillna("").astype(str).apply(preprocess_text).tolist()
    real_noisy_raw = [apply_noise(t, rng, level=0.35) for t in real_df["text"].fillna("").astype(str)]
    real_noisy_texts = [preprocess_text(t) for t in real_noisy_raw]

    real_lbl = real_df["label"].astype(int).values
    real_tech = encode_techniques(real_df["techniques"])
    real_act = real_df["action"].astype(int).values

    # 2. Generate synthetic dataset + apply noise to ~30% of rows
    synth_df = generate_synthetic_dataset(target_rows=6000, seed=SEED)
    is_noisy_flags = []
    aug_texts = []
    for txt in synth_df["text"].tolist():
        if rng.random() < 0.30:
            aug_texts.append(apply_noise(txt, rng, level=0.35))
            is_noisy_flags.append(True)
        else:
            aug_texts.append(txt)
            is_noisy_flags.append(False)

    synth_df["text"] = aug_texts
    synth_df["is_noisy"] = is_noisy_flags

    os.makedirs("data", exist_ok=True)
    synth_df.to_csv("data/synthetic.csv", index=False)
    print(f"Saved data/synthetic.csv: shape={synth_df.shape}, noisy={synth_df['is_noisy'].mean()*100:.1f}%")

    synth_clean_proc = synth_df["text"].apply(preprocess_text).tolist()
    synth_lbl = synth_df["label"].astype(int).values
    synth_tech = encode_techniques(synth_df["techniques"])
    synth_act = synth_df["action"].astype(int).values
    synth_groups = synth_df["family_id"].values

    # ==================== STEP 4: HONEST VALIDATION ====================
    print("\n" + "=" * 70)
    print("STEP 4: HONEST VALIDATION (CLEAN vs NOISY)")
    print("=" * 70)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

    # (a) Phase 1 baseline (86 real rows only) on 5-fold CV
    a_cln_lbl, a_cln_tech, a_cln_act = np.zeros(len(real_df), int), [""] * len(real_df), np.zeros(len(real_df), int)
    a_nsy_lbl, a_nsy_tech, a_nsy_act = np.zeros(len(real_df), int), [""] * len(real_df), np.zeros(len(real_df), int)
    for tr_i, va_i in skf.split(real_clean_texts, real_lbl):
        pl, pt, pa, _ = fit_and_predict(
            [real_clean_texts[i] for i in tr_i], real_lbl[tr_i], real_tech[tr_i], real_act[tr_i],
            [real_clean_texts[i] for i in va_i]
        )
        for k, idx in enumerate(va_i):
            a_cln_lbl[idx], a_cln_tech[idx], a_cln_act[idx] = pl[k], pt[k], pa[k]

        pl_n, pt_n, pa_n, _ = fit_and_predict(
            [real_clean_texts[i] for i in tr_i], real_lbl[tr_i], real_tech[tr_i], real_act[tr_i],
            [real_noisy_texts[i] for i in va_i]
        )
        for k, idx in enumerate(va_i):
            a_nsy_lbl[idx], a_nsy_tech[idx], a_nsy_act[idx] = pl_n[k], pt_n[k], pa_n[k]

    fa_a_c, fb_a_c, fc_a_c = compute_metrics(real_lbl, real_tech, real_act, a_cln_lbl, a_cln_tech, a_cln_act)
    fa_a_n, fb_a_n, fc_a_n = compute_metrics(real_lbl, real_tech, real_act, a_nsy_lbl, a_nsy_tech, a_nsy_act)
    print(f"(a) Real-only 5-fold CV  | CLEAN: A={fa_a_c:.4f}, B={fb_a_c:.4f}, C={fc_a_c:.4f} | NOISY: A={fa_a_n:.4f}, B={fb_a_n:.4f}, C={fc_a_n:.4f}")

    # (b) Train on synthetic only -> evaluate on all 86 real rows
    b_cln_lbl, b_cln_tech, b_cln_act, _ = fit_and_predict(synth_clean_proc, synth_lbl, synth_tech, synth_act, real_clean_texts)
    b_nsy_lbl, b_nsy_tech, b_nsy_act, _ = fit_and_predict(synth_clean_proc, synth_lbl, synth_tech, synth_act, real_noisy_texts)
    fa_b_c, fb_b_c, fc_b_c = compute_metrics(real_lbl, real_tech, real_act, b_cln_lbl, b_cln_tech, b_cln_act)
    fa_b_n, fb_b_n, fc_b_n = compute_metrics(real_lbl, real_tech, real_act, b_nsy_lbl, b_nsy_tech, b_nsy_act)
    print(f"(b) Synth-only -> Real   | CLEAN: A={fa_b_c:.4f}, B={fb_b_c:.4f}, C={fc_b_c:.4f} | NOISY: A={fa_b_n:.4f}, B={fb_b_n:.4f}, C={fc_b_n:.4f}")

    # (c) 5-fold CV on real rows + ALL synthetic in train fold (real weight 5x)
    c_cln_lbl, c_cln_tech, c_cln_act = np.zeros(len(real_df), int), [""] * len(real_df), np.zeros(len(real_df), int)
    c_nsy_lbl, c_nsy_tech, c_nsy_act = np.zeros(len(real_df), int), [""] * len(real_df), np.zeros(len(real_df), int)
    for tr_i, va_i in skf.split(real_clean_texts, real_lbl):
        tr_texts = [real_clean_texts[i] for i in tr_i] + synth_clean_proc
        tr_lbl = np.concatenate([real_lbl[tr_i], synth_lbl])
        tr_tech = np.vstack([real_tech[tr_i], synth_tech])
        tr_act = np.concatenate([real_act[tr_i], synth_act])
        weights = np.concatenate([np.full(len(tr_i), 5.0), np.ones(len(synth_lbl))])

        pl, pt, pa, _ = fit_and_predict(tr_texts, tr_lbl, tr_tech, tr_act, [real_clean_texts[i] for i in va_i], sample_weight=weights)
        for k, idx in enumerate(va_i):
            c_cln_lbl[idx], c_cln_tech[idx], c_cln_act[idx] = pl[k], pt[k], pa[k]

        pl_n, pt_n, pa_n, _ = fit_and_predict(tr_texts, tr_lbl, tr_tech, tr_act, [real_noisy_texts[i] for i in va_i], sample_weight=weights)
        for k, idx in enumerate(va_i):
            c_nsy_lbl[idx], c_nsy_tech[idx], c_nsy_act[idx] = pl_n[k], pt_n[k], pa_n[k]

    fa_c_c, fb_c_c, fc_c_c = compute_metrics(real_lbl, real_tech, real_act, c_cln_lbl, c_cln_tech, c_cln_act)
    fa_c_n, fb_c_n, fc_c_n = compute_metrics(real_lbl, real_tech, real_act, c_nsy_lbl, c_nsy_tech, c_nsy_act)
    print(f"(c) Real(5x)+Synth CV    | CLEAN: A={fa_c_c:.4f}, B={fb_c_c:.4f}, C={fc_c_c:.4f} | NOISY: A={fa_c_n:.4f}, B={fb_c_n:.4f}, C={fc_c_n:.4f}")

    # Per-class F1 & Confusion Matrix for (c) Clean
    per_cls_c = f1_score(real_lbl, c_cln_lbl, average=None)
    print("\nPer-class F1 for (c) CLEAN:", {k: round(float(per_cls_c[k]), 4) for k in range(6)})
    cm_c = confusion_matrix(real_lbl, c_cln_lbl)
    print("Confusion Matrix for (c) CLEAN (rows=true, cols=pred):\n", cm_c)
    per_cls_n = f1_score(real_lbl, c_nsy_lbl, average=None)
    print("Per-class F1 for (c) NOISY:", {k: round(float(per_cls_n[k]), 4) for k in range(6)})

    # (d) StratifiedGroupKFold on synthetic data grouped by family_id
    sgkf = StratifiedGroupKFold(n_splits=5)
    d_cln_lbl, d_cln_tech, d_cln_act = np.zeros(len(synth_df), int), [""] * len(synth_df), np.zeros(len(synth_df), int)
    d_nsy_lbl, d_nsy_tech, d_nsy_act = np.zeros(len(synth_df), int), [""] * len(synth_df), np.zeros(len(synth_df), int)
    synth_all_noisy_proc = [preprocess_text(apply_noise(t, rng, level=0.35)) for t in synth_df["text"].tolist()]

    for tr_i, va_i in sgkf.split(synth_clean_proc, synth_lbl, synth_groups):
        pl, pt, pa, _ = fit_and_predict(
            [synth_clean_proc[i] for i in tr_i], synth_lbl[tr_i], synth_tech[tr_i], synth_act[tr_i],
            [synth_clean_proc[i] for i in va_i]
        )
        for k, idx in enumerate(va_i):
            d_cln_lbl[idx], d_cln_tech[idx], d_cln_act[idx] = pl[k], pt[k], pa[k]

        pl_n, pt_n, pa_n, _ = fit_and_predict(
            [synth_clean_proc[i] for i in tr_i], synth_lbl[tr_i], synth_tech[tr_i], synth_act[tr_i],
            [synth_all_noisy_proc[i] for i in va_i]
        )
        for k, idx in enumerate(va_i):
            d_nsy_lbl[idx], d_nsy_tech[idx], d_nsy_act[idx] = pl_n[k], pt_n[k], pa_n[k]

    fa_d_c, fb_d_c, fc_d_c = compute_metrics(synth_lbl, synth_tech, synth_act, d_cln_lbl, d_cln_tech, d_cln_act)
    fa_d_n, fb_d_n, fc_d_n = compute_metrics(synth_lbl, synth_tech, synth_act, d_nsy_lbl, d_nsy_tech, d_nsy_act)
    print(f"\n(d) Synth GroupKFold     | CLEAN: A={fa_d_c:.4f}, B={fb_d_c:.4f}, C={fc_d_c:.4f} | NOISY: A={fa_d_n:.4f}, B={fb_d_n:.4f}, C={fc_d_n:.4f}")

    # ==================== STEP 5: RETRAIN & PREDICT ====================
    print("\n" + "=" * 70)
    print("STEP 5: RETRAIN PHASE 2 MODEL & PREDICT test.csv")
    print("=" * 70)
    full_texts = real_clean_texts + real_noisy_texts + synth_clean_proc
    full_lbl = np.concatenate([real_lbl, real_lbl, synth_lbl])
    full_tech = np.vstack([real_tech, real_tech, synth_tech])
    full_act = np.concatenate([real_act, real_act, synth_act])
    full_weights = np.concatenate([
        np.full(len(real_lbl), 5.0),
        np.full(len(real_lbl), 3.0),
        np.ones(len(synth_lbl)),
    ])

    char_vec, word_vec, X_full = fit_vectorizers(full_texts)
    full_clf_a = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED).fit(X_full, full_lbl, sample_weight=full_weights)
    full_clf_b = OneVsRestClassifier(LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)).fit(X_full, full_tech)
    full_clf_c = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED).fit(X_full, full_act, sample_weight=full_weights)

    test_path = os.path.join(data_dir, "test.csv")
    out_path = "submission_phase2.csv"
    first_chunk = True
    label_counts = {k: 0 for k in range(6)}
    total_rows = 0

    for chunk in pd.read_csv(test_path, chunksize=50000):
        clean_batch = chunk["text"].fillna("").astype(str).apply(preprocess_text).tolist()
        X_test = transform_vectorizers(char_vec, word_vec, clean_batch)
        probs_a = full_clf_a.predict_proba(X_test)
        preds_a = np.argmax(probs_a, axis=1)
        confs = np.max(probs_a, axis=1)
        tech_probs = full_clf_b.predict_proba(X_test)
        preds_c = full_clf_c.predict(X_test)

        preds_a, tech_strs, preds_c, confs = postprocess_predictions(preds_a, tech_probs, preds_c, confs)
        for lbl in preds_a:
            label_counts[int(lbl)] += 1

        pd.DataFrame({
            "id": chunk["id"].values.astype(int),
            "label": preds_a.astype(int),
            "techniques": tech_strs,
            "action": preds_c.astype(int),
            "confidence": confs,
        }).to_csv(out_path, mode="w" if first_chunk else "a", header=first_chunk, index=False)
        first_chunk = False
        total_rows += len(chunk)

    print(f"Saved {out_path} ({total_rows:,} rows) in {time.time() - t0:.2f}s")
    print("Phase 2 Predicted Label Distribution on test.csv:")
    for k, v in label_counts.items():
        print(f"  Label {k}: {v:7,d} ({v/total_rows*100:5.2f}%)")

    print("\nValidating submission_phase2.csv:")
    validate_submission(out_path)

    # ==================== STEP 6: QUALITY CHECK (40 SAMPLES) ====================
    print("\n" + "=" * 70)
    print("STEP 6: QUALITY CHECK (40 GENERATED ROWS)")
    print("=" * 70)
    sampled_idx = set()
    for lang in ["english", "bangla", "banglish", "mixed"]:
        sampled_idx.update(synth_df[synth_df["language"] == lang].sample(5, random_state=SEED).index)
    for lbl in range(6):
        sampled_idx.update(synth_df[synth_df["label"] == lbl].sample(3, random_state=SEED).index)
    sampled_idx.update(synth_df[synth_df["n_turns"] > 1].sample(5, random_state=SEED).index)
    sampled_idx.update(synth_df[synth_df["is_hard_negative"]].sample(5, random_state=SEED).index)
    sampled_idx.update(synth_df[synth_df["is_noisy"]].sample(5, random_state=SEED).index)

    qc_list = sorted(list(sampled_idx))[:40]
    for i, idx in enumerate(qc_list):
        r = synth_df.loc[idx]
        print(f"[{i+1:02d}] L={r['label']} | T={r['techniques']:8s} | A={r['action']} | {r['language']:8s} | turns={r['n_turns']} | hard={r['is_hard_negative']} | noisy={r['is_noisy']}")
        print(f"     {repr(r['text'][:140])}")


if __name__ == "__main__":
    run_phase2()

Saved data/synthetic.csv: shape=(6000, 10), noisy=29.8%

STEP 4: HONEST VALIDATION (CLEAN vs NOISY)


(a) Real-only 5-fold CV  | CLEAN: A=0.5828, B=0.4463, C=0.4940 | NOISY: A=0.5624, B=0.4162, C=0.4205


(b) Synth-only -> Real   | CLEAN: A=0.9474, B=0.8637, C=0.9255 | NOISY: A=0.9372, B=0.8727, C=0.9335


(c) Real(5x)+Synth CV    | CLEAN: A=0.9372, B=0.8598, C=0.9335 | NOISY: A=0.9466, B=0.8648, C=0.9335

Per-class F1 for (c) CLEAN: {0: 0.9206, 1: 0.8333, 2: 0.9524, 3: 1.0, 4: 1.0, 5: 0.9167}
Confusion Matrix for (c) CLEAN (rows=true, cols=pred):
 [[29  2  0  0  0  1]
 [ 1 10  1  0  0  0]
 [ 0  0 10  0  0  0]
 [ 0  0  0 11  0  0]
 [ 0  0  0  0  9  0]
 [ 1  0  0  0  0 11]]
Per-class F1 for (c) NOISY: {0: 0.9375, 1: 0.8333, 2: 0.9524, 3: 1.0, 4: 1.0, 5: 0.9565}



(d) Synth GroupKFold     | CLEAN: A=0.5887, B=0.4150, C=0.5505 | NOISY: A=0.5595, B=0.3903, C=0.5028

STEP 5: RETRAIN PHASE 2 MODEL & PREDICT test.csv


Saved submission_phase2.csv (500,000 rows) in 157.96s
Phase 2 Predicted Label Distribution on test.csv:
  Label 0: 201,976 (40.40%)
  Label 1:  92,702 (18.54%)
  Label 2:  41,790 ( 8.36%)
  Label 3:  41,749 ( 8.35%)
  Label 4:  59,526 (11.91%)
  Label 5:  62,257 (12.45%)

Validating submission_phase2.csv:


PASS

STEP 6: QUALITY CHECK (40 GENERATED ROWS)
[01] L=1 | T=T3|T5    | A=2 | mixed    | turns=1 | hard=True | noisy=False
     'Bhai ekta bhul hoye geche. Amar 1,000 taka vul kore apnar ChakaPay e chole gese. Please ferot den 018XXXXXX44 e. Ami bipode.'
[02] L=3 | T=T3|T6    | A=2 | english  | turns=1 | hard=False | noisy=True
     'Double yor moneyin 7 days! ІnEv5t Tk 250, getdоUB\u200blе guaranteed. Send via Cellfin 018XXXXXX45.'
[03] L=2 | T=T1|T4|T5 | A=3 | bangla   | turns=3 | hard=False | noisy=True
     'C\u200baller: অভিনন্দন! আপনিbOngonEtলটারিতে 2,\u200c000 টাকা জিতেছেন। ||| You:সত্যি নাক\u200bি? কী\u200cভ াবে পাবো? ||| Caller:  এই  লিংকে  গিয়ে  ফর্ম  পূরণ  কর'
[04] L=4 | T=T1|T3|T5 | A=2 | english  | turns=5 | hard=True | noisy=True
     "A: Hello, this is Karatoa. You have a p4ckаg3 #PF50508 frm abroad. ||| B: I d\u200cId'T order anything. ||| A: It is registered under your name a"
[05] L=0 | T=NONE     | A=0 | banglish | turns=1 | hard=True | noisy=True
     'Dhonyobad! Ap